In [23]:
# IMPORTS
import torch
import torch.nn as nn
from Binn import BINN
import data_handling as dh
import binn_training as bt
import os 
import custom_train_test_split as ctts
import pandas as pd, numpy as np

In [24]:
# GLOBALS
ALL_CELLTYPES = [0,1,2,3,4,5,6,7,8]
EPOCHS = 20
TRAIN_SIZE = 0.8
BATCH_SIZE = 16

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

base_path = "/data/shared/alzgene26"
data_path = base_path + "/data/processed_data/completed/mg_200_mc_200_mhvg1000/"
comp_proc_data_path = base_path + "/data/processed_data/extracted_from_completed/"
comp_proc_data_path

'/data/shared/alzgene26/data/processed_data/extracted_from_completed/'

In [25]:
print("Reading completed processed adata...")
datasets = ctts.read_files(to_include=ALL_CELLTYPES, filepath=comp_proc_data_path)

Reading completed processed adata...
Labels to include: ['astro', 'exc1', 'exc2', 'exc3', 'immune', 'inhi', 'oligo', 'opcs', 'vasc']
Reading astro
Reading exc1
Reading exc2
Reading exc3
Reading immune
Reading inhi
Reading oligo
Reading opcs
Reading vasc


In [42]:
print("Transposing datasets...")
datasets_t = {}
for celltype in datasets.keys():
    adata_t = datasets[celltype].T
    datasets_t[celltype] = adata_t

datasets_t["astro"]

Transposing datasets...


AnnData object with n_obs × n_vars = 1000 × 1234
    obs: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'mean', 'std'
    var: 'subject', 'cell_type_high_resolution', 'n_obs_aggregated', 'AD_status'
    uns: 'log1p'
    layers: 'sum'

In [41]:
print("Creating AnnCollection...")
acollection = ctts.create_encoded_collection(datasets_t)
acollection
# NOTE: join_vars="inner" because "outer" would create artificial zeroes in gene expresion???

Creating AnnCollection...


/data/users/thomath/.venv/lib/python3.12/site-packages/legacy_api_wrap/__init__.py:88: UserWarning: Observation names are not unique.
  return fn(*args_all, **kw)


KeyError: 'cell_type_high_res'

In [17]:
print("Creating train/test split...")
train_adata, test_adata = ctts.custom_train_test_split(acollection, train_size=TRAIN_SIZE)

Creating train/test split...
Train Subjects: 341
Test Subjects: 86


In [18]:
print("Getting dataloaders...")
train_loader, test_loader = dh.create_dataloaders(train_adata, test_adata)

Getting dataloaders...


In [19]:
mask_paths = [f"/PathwayData/MaskMatrixLayers/mg_200_mc_200_mhvg1000/oligo_exc3_exc2_vasc_immune_astro_inhi_opcs_exc1_layer_{i}_mask.csv" 
              for i in range(5)]

# Mask matrix files
masks = {}
for i, mask_path in enumerate(mask_paths):
    df = pd.read_csv(base_path + mask_path, index_col=0)
    masks.update({f"df{i}": df})
    print(f"Matrix {i} shape: {df.shape}")

print(masks.keys())

Matrix 0 shape: (8075, 1160)
Matrix 1 shape: (1160, 551)
Matrix 2 shape: (551, 188)
Matrix 3 shape: (188, 29)
Matrix 4 shape: (29, 1)
dict_keys(['df0', 'df1', 'df2', 'df3', 'df4'])


In [20]:
# Extract pure number representation matrices from masks
mask_matrix_list = [masks[mask].to_numpy() for mask in masks]

# Starting amount of features
in_features = masks["df0"].shape[0]

# Extract layer dimensions
layers_list = [masks[mask].shape[1] for mask in masks]

print(f"input features: {in_features}")
print(f"layer list: {layers_list}")

input features: 8075
layer list: [1160, 551, 188, 29, 1]


In [21]:
# Conversion for mask matrix list, creates tensors for BINN, transposed
tensor_masks = [torch.tensor(mask).float().t() for mask in mask_matrix_list]
# Put on device for BINN
tensor_masks = [mask.to(device) for mask in tensor_masks]

In [22]:
model = BINN(in_features=in_features,
                  layers_list=layers_list,
                  mask_list=tensor_masks).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [23]:
# Step through the training
for epoch in range(3):
    train_loss, train_acc = bt.train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = bt.test_one_epoch(model, test_loader, criterion, device)
    
    print(f"Epoch {epoch+1} / {3}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Test Loss:  {test_loss:.4f} | Test Acc:  {test_acc:.4f}")
    print("-" * 30)


/data/users/thomath/kand/BINN/binn_training.py:24: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(batch.obs['AD_status']).float().reshape(-1, 1).to(device)


RuntimeError: mat1 and mat2 shapes cannot be multiplied (16x248 and 8075x1160)